# Stage 2: Batch Transformation Pipeline
## Sportlytics Athletics — Professional Basketball Analytics
Reads raw data from HDFS raw zone, cleans and standardizes, joins datasets,
computes aggregations to answer business questions, and writes Parquet outputs
to HDFS processed and analytics zones.ts

In [1]:
# Starting Spark session (local mode - avoids version mismatch with spark master)
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("Sportlytics-Stage2-Transforms")
    .master("spark://spark-master:7077")
    .config("spark.executor.memory", "1g")
    .config("spark.driver.memory", "1g")
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("SparkSession ready!")

Spark version: 3.5.0
SparkSession ready!


## Part 1: Load Raw Data from HDFS

In [2]:
# Load CSV files from HDFS raw zone
player_tracking = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("hdfs://namenode:9000/sportlytics/raw/player-tracking.csv.gz")
)

injury_reports = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("hdfs://namenode:9000/sportlytics/raw/injury-reports.csv.gz")
)

team_schedules = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("hdfs://namenode:9000/sportlytics/raw/team-schedules.csv.gz")
)

# Load JSON files from HDFS raw zone
game_stats = spark.read.json("hdfs://namenode:9000/sportlytics/raw/game-stats.json.gz")
training_sessions = spark.read.json("hdfs://namenode:9000/sportlytics/raw/training-sessions.json.gz")

print("All datasets loaded!")

All datasets loaded!


In [3]:
# Print schemas and row counts for all datasets
for name, df in [
    ("player_tracking", player_tracking),
    ("game_stats", game_stats),
    ("injury_reports", injury_reports),
    ("training_sessions", training_sessions),
    ("team_schedules", team_schedules)
]:
    print(f"\n--- {name} ---")
    print(f"Columns: {df.columns}")
    print(f"Row count: {df.count()}")
    df.printSchema()


--- player_tracking ---
Columns: ['game_id', 'player_id', 'timestamp', 'x_court_ft', 'y_court_ft', 'speed_mph', 'acceleration', 'distance_covered_ft', 'heart_rate_bpm', 'team_id']
Row count: 750000
root
 |-- game_id: string (nullable = true)
 |-- player_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- x_court_ft: double (nullable = true)
 |-- y_court_ft: double (nullable = true)
 |-- speed_mph: double (nullable = true)
 |-- acceleration: double (nullable = true)
 |-- distance_covered_ft: double (nullable = true)
 |-- heart_rate_bpm: integer (nullable = true)
 |-- team_id: string (nullable = true)


--- game_stats ---
Columns: ['assists', 'blocks', 'fouls', 'game_id', 'minutes_played', 'player_id', 'plus_minus', 'points', 'quarter', 'rebounds', 'shot_attempts', 'shot_made', 'steals', 'three_pt_attempts', 'three_pt_made', 'turnovers']
Row count: 300000
root
 |-- assists: long (nullable = true)
 |-- blocks: long (nullable = true)
 |-- fouls: long (nullable = 

## Part 2: Clean and Standardize Data

In [4]:
# Check null counts for all datasets
from pyspark.sql.functions import col, count, when, isnan

def null_check(df, name):
    print(f"\n--- {name} null counts ---")
    df.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in df.columns
    ]).show()

null_check(player_tracking, "player_tracking")
null_check(game_stats, "game_stats")
null_check(injury_reports, "injury_reports")
null_check(training_sessions, "training_sessions")
null_check(team_schedules, "team_schedules")


--- player_tracking null counts ---
+-------+---------+---------+----------+----------+---------+------------+-------------------+--------------+-------+
|game_id|player_id|timestamp|x_court_ft|y_court_ft|speed_mph|acceleration|distance_covered_ft|heart_rate_bpm|team_id|
+-------+---------+---------+----------+----------+---------+------------+-------------------+--------------+-------+
|      0|        0|        0|         0|         0|        0|           0|                  0|             0|      0|
+-------+---------+---------+----------+----------+---------+------------+-------------------+--------------+-------+


--- game_stats null counts ---
+-------+------+-----+-------+--------------+---------+----------+------+-------+--------+-------------+---------+------+-----------------+-------------+---------+
|assists|blocks|fouls|game_id|minutes_played|player_id|plus_minus|points|quarter|rebounds|shot_attempts|shot_made|steals|three_pt_attempts|three_pt_made|turnovers|
+-------+---

In [5]:
#Inspecting stats
print("--- player_tracking stats ---")
player_tracking.select("heart_rate_bpm", "x_court_ft", "y_court_ft").describe().show()

print("--- training_sessions stats ---")
training_sessions.select("heart_rate_avg", "heart_rate_max", "intensity_score", "sleep_hours_prev_night").describe().show()

print("--- team_schedules stats ---")
team_schedules.select("travel_distance_miles", "rest_days_since_last_game").describe().show()

--- player_tracking stats ---
+-------+------------------+------------------+------------------+
|summary|    heart_rate_bpm|        x_court_ft|        y_court_ft|
+-------+------------------+------------------+------------------+
|  count|            750000|            750000|            750000|
|   mean|144.54029066666666| 46.96971106666719|24.951655999999783|
| stddev|24.973512372724397|27.150410572467884|14.417910581837292|
|    min|                60|               0.0|               0.0|
|    max|               220|              94.0|              50.0|
+-------+------------------+------------------+------------------+

--- training_sessions stats ---
+-------+-----------------+------------------+------------------+----------------------+
|summary|   heart_rate_avg|    heart_rate_max|   intensity_score|sleep_hours_prev_night|
+-------+-----------------+------------------+------------------+----------------------+
|  count|           150000|            150000|            150000|  

In [6]:
# Check exact count of out-of-range heart rates
print("player_tracking - heart rate out of range (< 60 or > 220):")
player_tracking.filter(
    (F.col("heart_rate_bpm") < 60) | (F.col("heart_rate_bpm") > 220)
).count()

print("training_sessions - heart_rate_avg out of range:")
training_sessions.filter(
    (F.col("heart_rate_avg") < 60) | (F.col("heart_rate_avg") > 220)
).count()

print("training_sessions - heart_rate_max out of range:")
training_sessions.filter(
    (F.col("heart_rate_max") < 60) | (F.col("heart_rate_max") > 220)
).count()

player_tracking - heart rate out of range (< 60 or > 220):
training_sessions - heart_rate_avg out of range:
training_sessions - heart_rate_max out of range:


0

 ── Data Cleaning Summary ─────────────────────────────────────────

 After thorough inspection:
 - 0 null values across all 5 datasets
 - 0 out-of-range heart rate values in player_tracking or training_sessions
 - Court coordinates within valid bounds (0-94ft x 0-50ft)
 - Travel distances and rest days are non-negative
 - Data was synthetically generated with high quality

In [7]:
#Data Cleaning Steps

# Required transformation 1: cast training_sessions date from string to DateType
training_sessions_clean = training_sessions.withColumn(
    "date", F.to_date(F.col("date"))
)

# Required transformation 2: handle missing fields in injury_reports defensively
injury_reports_clean = injury_reports.fillna({
    "games_missed": 0,
    "treatment": "unknown",
    "severity": "unknown"
})

# Remaining datasets are clean - aliased for pipeline consistency
player_tracking_clean = player_tracking
game_stats_clean = game_stats
team_schedules_clean = team_schedules

print("✓ Cleaning complete")
print(f"✓ training_sessions date cast: {training_sessions_clean.schema['date'].dataType}")
print("✓ injury_reports missing fields handled defensively")
print("✓ All other datasets validated and ready for joining")

✓ Cleaning complete
✓ training_sessions date cast: DateType()
✓ injury_reports missing fields handled defensively
✓ All other datasets validated and ready for joining


## Part 3: Joins

### Join 1: Player tracking with game stats (Fatigue-Performance)

In [10]:
# Join 1: Player tracking with game stats
# Links spatial exertion data with box score outcomes per player per game
tracking_with_stats = player_tracking_clean.join(
    game_stats_clean,
    on=["game_id", "player_id"],
    how="inner"
)

print(f"player_tracking rows: {player_tracking_clean.count()}")
print(f"game_stats rows: {game_stats_clean.count()}")
print(f"tracking_with_stats rows: {tracking_with_stats.count()}")
tracking_with_stats.printSchema()
tracking_with_stats.show(5)
tracking_with_stats.explain()

player_tracking rows: 750000
game_stats rows: 300000
tracking_with_stats rows: 250379
root
 |-- game_id: string (nullable = true)
 |-- player_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- x_court_ft: double (nullable = true)
 |-- y_court_ft: double (nullable = true)
 |-- speed_mph: double (nullable = true)
 |-- acceleration: double (nullable = true)
 |-- distance_covered_ft: double (nullable = true)
 |-- heart_rate_bpm: integer (nullable = true)
 |-- team_id: string (nullable = true)
 |-- assists: long (nullable = true)
 |-- blocks: long (nullable = true)
 |-- fouls: long (nullable = true)
 |-- minutes_played: double (nullable = true)
 |-- plus_minus: long (nullable = true)
 |-- points: long (nullable = true)
 |-- quarter: long (nullable = true)
 |-- rebounds: long (nullable = true)
 |-- shot_attempts: long (nullable = true)
 |-- shot_made: long (nullable = true)
 |-- steals: long (nullable = true)
 |-- three_pt_attempts: long (nullable = true)
 |-- thre

##### Notes

- 750k tracking rows + 300k game_stats rows = 250k joined rows — this makes sense because not every tracking timestamp matches every quarter record, it's a many-to-many relationship
- The physical plan shows Spark used a BroadcastHashJoin automatically — which is efficient since game_stats is the smaller table
- The schema looks correct with all columns from both datasets combined

### Join 2: Training sessions with team schedules (Training load)

In [11]:
# Initial attempt - join on date only
# Issue: causes row multiplication (150k → 436k) because multiple games per date
# Fix below uses team_id to correctly match player's team

training_with_schedule = training_sessions_clean.join(
    team_schedules_clean,
    on="date",
    how="left"
)

print(f"training_sessions rows: {training_sessions_clean.count()}")
print(f"team_schedules rows: {team_schedules_clean.count()}")
print(f"training_with_schedule rows: {training_with_schedule.count()}")
training_with_schedule.printSchema()
training_with_schedule.show(5)

training_sessions rows: 150000
team_schedules rows: 10000
training_with_schedule rows: 436099
root
 |-- date: date (nullable = true)
 |-- calories_burned: long (nullable = true)
 |-- duration_min: long (nullable = true)
 |-- heart_rate_avg: long (nullable = true)
 |-- heart_rate_max: long (nullable = true)
 |-- intensity_score: long (nullable = true)
 |-- player_id: string (nullable = true)
 |-- session_type: string (nullable = true)
 |-- sleep_hours_prev_night: double (nullable = true)
 |-- game_id: string (nullable = true)
 |-- home_team: string (nullable = true)
 |-- away_team: string (nullable = true)
 |-- venue: string (nullable = true)
 |-- travel_distance_miles: integer (nullable = true)
 |-- rest_days_since_last_game: integer (nullable = true)
 |-- back_to_back: boolean (nullable = true)
 |-- result: string (nullable = true)
 |-- score_home: integer (nullable = true)
 |-- score_away: integer (nullable = true)

+----------+---------------+------------+--------------+------------

In [12]:
# Corrected Join 2: training sessions with team schedules

# Step 1: derive team_id for each player from tracking data
# (training_sessions has no team_id column)
player_team_map = player_tracking_clean.select("player_id", "team_id").distinct()

# Step 2: add team_id to training sessions
training_sessions_with_team = training_sessions_clean.join(
    player_team_map,
    on="player_id",
    how="left"
)

# Step 3: join with team_schedules on both date AND team
# ensures each session matches only games where the player's team played
training_with_schedule = training_sessions_with_team.join(
    team_schedules_clean,
    on=(
        (training_sessions_with_team.date == team_schedules_clean.date) &
        (
            (training_sessions_with_team.team_id == team_schedules_clean.home_team) |
            (training_sessions_with_team.team_id == team_schedules_clean.away_team)
        )
    ),
    how="left"
).drop(team_schedules_clean.date)

print(f"training_sessions rows: {training_sessions_clean.count()}")
print(f"training_with_schedule rows: {training_with_schedule.count()}")
training_with_schedule.show(5)

training_sessions rows: 150000
training_with_schedule rows: 154061
+---------+---------------+----------+------------+--------------+--------------+---------------+------------+----------------------+-------+-------+---------+---------+-----+---------------------+-------------------------+------------+------+----------+----------+
|player_id|calories_burned|      date|duration_min|heart_rate_avg|heart_rate_max|intensity_score|session_type|sleep_hours_prev_night|team_id|game_id|home_team|away_team|venue|travel_distance_miles|rest_days_since_last_game|back_to_back|result|score_home|score_away|
+---------+---------------+----------+------------+--------------+--------------+---------------+------------+----------------------+-------+-------+---------+---------+-----+---------------------+-------------------------+------------+------+----------+----------+
| PLR-0321|            914|2026-02-20|          86|           179|           218|             10|    practice|                   8.1|TE

### Join 3: Player tracking+stats with team schedules

In [15]:
# Adds schedule context (back-to-back, travel distance, rest days) to player performance data
# Enables analysis of back-to-back impact (Q3) and travel/rest interaction (Q4)
tracking_stats_schedule = tracking_with_stats.join(
    team_schedules_clean,
    on="game_id",
    how="left"
)

print(f"tracking_with_stats rows: {tracking_with_stats.count()}")
print(f"tracking_stats_schedule rows: {tracking_stats_schedule.count()}")
tracking_stats_schedule.printSchema()
tracking_stats_schedule.show(5)

tracking_with_stats rows: 250379
tracking_stats_schedule rows: 250379
root
 |-- game_id: string (nullable = true)
 |-- player_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- x_court_ft: double (nullable = true)
 |-- y_court_ft: double (nullable = true)
 |-- speed_mph: double (nullable = true)
 |-- acceleration: double (nullable = true)
 |-- distance_covered_ft: double (nullable = true)
 |-- heart_rate_bpm: integer (nullable = true)
 |-- team_id: string (nullable = true)
 |-- assists: long (nullable = true)
 |-- blocks: long (nullable = true)
 |-- fouls: long (nullable = true)
 |-- minutes_played: double (nullable = true)
 |-- plus_minus: long (nullable = true)
 |-- points: long (nullable = true)
 |-- quarter: long (nullable = true)
 |-- rebounds: long (nullable = true)
 |-- shot_attempts: long (nullable = true)
 |-- shot_made: long (nullable = true)
 |-- steals: long (nullable = true)
 |-- three_pt_attempts: long (nullable = true)
 |-- three_pt_made: long 